In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import types
import pandas as pd
from pyspark.sql.types import StructType, StructField, IntegerType, TimestampType, StringType, DoubleType, LongType
import pathlib
from pyspark.sql import functions as F 


In [ ]:
from pathlib import Path

DATA_DIR = Path.home() / 'spark-zoomcamp' / 'data'

In [ ]:
spark = SparkSession.builder.master('local[*]').appName('union_data').getOrCreate()

In [ ]:
path_pq = '/home/ilyes/spark-zoomcamp/data/pq'

In [ ]:
df_green = spark.read.parquet(f'{path_pq}/green/*/*')
df_yellow = spark.read.parquet(f'{path_pq}/yellow/*/*')

In [ ]:
df_green.printSchema()

In [ ]:
df_yellow.count()

In [ ]:
df_yellow.printSchema()

In [ ]:
df_yellow = df_yellow.withColumnsRenamed({'tpep_pickup_datetime':'pickup_datetime','tpep_dropoff_datetime':'dropoff_datetime'})

In [ ]:
df_green = df_green.withColumnsRenamed({'lpep_pickup_datetime':'pickup_datetime','lpep_dropoff_datetime':'dropoff_datetime'})

In [ ]:
df_green.printSchema()

In [ ]:
columns_all = []

yellow_columns = set(df_yellow.columns)
for col in df_green.columns : 
    if col in yellow_columns:
        columns_all.append(col)


In [ ]:
print(columns_all)

In [ ]:
df_green_n = df_green.select(columns_all).withColumn('service_type',F.lit('green'))

In [ ]:
df_green_n.printSchema()

In [ ]:
df_yellow_n = df_yellow.select(columns_all).withColumn('service_type',F.lit('yellow'))

In [ ]:
trips_union_df = df_yellow_n.unionByName(df_green_n)

In [ ]:
trips_union_df.groupBy('service_type').count().show()

In [ ]:
trips_union_df.registerTempTable('trips_data')

In [ ]:
spark.sql("""SELECT service_type, count(*)
FROM trips_data
group by service_type
""").show()

In [ ]:
df_result = spark.sql("""
SELECT
    -- Revenue grouping
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month,
    service_type,

    -- Revenue calculation
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [ ]:
# df_result.show()

In [ ]:
df_result.write.parquet(str(DATA_DIR / 'report'/'revenue'/'green'), mode='overwrite')


In [ ]:
df_result.coalesce(1).write.parquet(str(DATA_DIR / 'report'/'revenue'/'green'),mode='overwrite')

## GroupBy

For Green taxi

In [ ]:
df_green.createOrReplaceTempView('green')

In [ ]:
df_res = spark.sql("""
SELECT
    date_trunc('hour',pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    (pickup_datetime >= '2020-01-01') and (pickup_datetime < '2022-01-01')
GROUP BY
    1, 2
ORDER BY
    hour, zone






""")

In [ ]:
df_res.coalesce(1).write.parquet(str(DATA_DIR / 'report'/'revenue'/'green'), mode='overwrite')


In [ ]:
df_res.write.parquet(str(DATA_DIR / 'report'/'revenue'/'green'), mode='overwrite')


In [ ]:
df_res.show()

For yellow Taxi

In [ ]:
df_yellow.createOrReplaceTempView('yellow')

In [ ]:
df_revenue_yellow = spark.sql(""" 
SELECT
    date_trunc('hour',pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    (pickup_datetime >= '2020-01-01') and (pickup_datetime < '2022-01-01')
GROUP BY
    1, 2
ORDER BY
    hour, zone




""")

In [ ]:
df_revenue_yellow.write.parquet(str(DATA_DIR/'report'/'revenue'/'yellow'),mode='overwrite')

In [ ]:
df_revenue_yellow.coalesce(1).write.parquet(str(DATA_DIR/'report'/'revenue'/'yellow'),mode='overwrite')

In [ ]:
df_green_revenu = spark.read.parquet(str(DATA_DIR/'report'/'revenue'/'green'))

In [ ]:
df_yellow_revenu = spark.read.parquet(str(DATA_DIR/'report'/'revenue'/'yellow'))

In [ ]:
df_yellow_revenu.printSchema()

In [ ]:
df_yellow_revenu = df_yellow_revenu.withColumnRenamed('amount','yellow_amount')
df_green_revenu = df_green_revenu.withColumnRenamed('amount','green_amount')
df_yellow_revenu = df_yellow_revenu.withColumnRenamed('number_records','yellow_number_records')
df_green_revenu = df_green_revenu.withColumnRenamed('number_records','green_number_records')



In [ ]:
df_green_revenu.printSchema()

In [ ]:
df_yellow_revenu.printSchema()

In [ ]:
df_revenue_join = df_green_revenu.join(df_yellow_revenu,on=['hour','zone'],how='outer')

In [ ]:
df_revenue_join.printSchema()

In [ ]:
df_revenue_join.write.parquet(str(DATA_DIR/'report'/'revenue'/'total'),mode='overwrite')

In [ ]:
df_zone = spark.read.parquet(str(DATA_DIR/'zones'))

In [ ]:
df_revenue_join.printSchema()

In [ ]:
df_zone.printSchema()

In [ ]:
df_result_zone = df_zone.join(df_revenue_join, df_revenue_join.zone == df_zone.LocationID)

In [ ]:
df_result_zone.show()

In [ ]:
df_green.count()